# EMG-ASL Data Exploration

This notebook provides an end-to-end visual inspection of the raw EMG dataset collected
for the ASL recognition project.

**Goals**
- Understand the class distribution across participants and signs.
- Visualise raw 8-channel EMG signals before and after filtering.
- Inspect feature distributions per ASL sign to assess separability.
- Identify potential data quality issues (noise, class imbalance, missing channels).

**Dataset format**  
Each session CSV has columns: `timestamp_ms, ch1, ch2, ch3, ch4, ch5, ch6, ch7, ch8, label`.  
Files are stored under `data/raw/` and named `{participant_id}_{session_date}.csv`.

In [ ]:
# ─── Imports & path setup ───────────────────────────────────────────────────
import sys
import os
from pathlib import Path

# Ensure the repository root is on sys.path so that `src` is importable
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from src.data.loader import load_dataset, create_windows
from src.utils.filters import apply_full_filter_chain
from src.utils.features import extract_features, get_feature_names
from src.utils.constants import (
    ASL_LABELS,
    N_CHANNELS,
    SAMPLE_RATE,
    WINDOW_SIZE_MS,
    FEATURE_VECTOR_SIZE,
)

# Plotting defaults
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

DATA_RAW_DIR = REPO_ROOT / "data" / "raw"
CHANNEL_COLS = [f"ch{i+1}" for i in range(N_CHANNELS)]
FS = float(SAMPLE_RATE)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_RAW_DIR}")
print(f"ASL vocabulary  : {len(ASL_LABELS)} labels")

In [ ]:
# ─── Load dataset ───────────────────────────────────────────────────────────
if not DATA_RAW_DIR.exists() or not any(DATA_RAW_DIR.glob("*.csv")):
    # Generate synthetic data for demonstration purposes
    print("No real data found in data/raw/ — generating synthetic dataset.")

    rng = np.random.default_rng(0)
    n_participants = 3
    n_signs_per_label = 10
    samples_per_sign = int(FS * 2.0)   # 2 s @ 200 Hz

    rows = []
    for pid in range(1, n_participants + 1):
        t_cursor = 0.0
        for label in ASL_LABELS[:10]:   # first 10 labels for brevity
            for _ in range(n_signs_per_label):
                for s in range(samples_per_sign):
                    ts = t_cursor + s / FS * 1000.0
                    channels = rng.standard_normal(N_CHANNELS) * 0.5 + rng.uniform(-0.1, 0.1, N_CHANNELS)
                    rows.append({"participant_id": f"P{pid:03d}", "timestamp_ms": ts,
                                 **{f"ch{i+1}": channels[i] for i in range(N_CHANNELS)},
                                 "label": label})
                t_cursor += samples_per_sign / FS * 1000.0

    df = pd.DataFrame(rows)
    print(f"Synthetic dataset shape: {df.shape}")
else:
    df = load_dataset(DATA_RAW_DIR)

print("\n=== Dataset summary ===")
print(f"Total samples      : {len(df):,}")
print(f"Participants       : {df['participant_id'].nunique()}")
print(f"Unique labels      : {df['label'].nunique()}")
print(f"Columns            : {list(df.columns)}")
print("\nClass distribution:")
print(df['label'].value_counts().to_string())

In [ ]:
# ─── Plot raw EMG signals — 8 channels, 5-second excerpt ────────────────────
# Select the first 5 seconds from the first participant session
first_pid = df['participant_id'].unique()[0]
session_df = df[df['participant_id'] == first_pid].copy().reset_index(drop=True)

t_start_ms = session_df['timestamp_ms'].min()
t_end_ms   = t_start_ms + 5_000.0   # 5 seconds
excerpt = session_df[(session_df['timestamp_ms'] >= t_start_ms) &
                     (session_df['timestamp_ms'] <= t_end_ms)]

t = (excerpt['timestamp_ms'].values - t_start_ms) / 1000.0   # → seconds

fig, axes = plt.subplots(N_CHANNELS, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Raw EMG — Participant {first_pid} (first 5 s)", fontsize=13, y=1.01)

colors = plt.cm.tab10(np.linspace(0, 1, N_CHANNELS))

for ch_idx, (ax, col) in enumerate(zip(axes, CHANNEL_COLS)):
    ax.plot(t, excerpt[col].values, lw=0.5, color=colors[ch_idx])
    ax.set_ylabel(f"Ch {ch_idx+1}\n(mV)", fontsize=8, labelpad=2)
    ax.tick_params(labelsize=7)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

axes[-1].set_xlabel("Time (s)", fontsize=10)
plt.tight_layout()
plt.show()
print(f"Plotted {len(excerpt)} samples ({t[-1]:.2f} s excerpt).")

In [ ]:
# ─── Visualise signal through each processing step ──────────────────────────
# Take 2 seconds of a single labelled sign as the example window
label_example = session_df['label'].dropna().unique()[0]
sign_mask = session_df['label'] == label_example
sign_df = session_df[sign_mask].iloc[:int(FS * 2.0)]  # first 2 s of that sign

raw_window = sign_df[CHANNEL_COLS].values.astype(np.float64)   # (T, 8)

from src.utils.filters import dc_remove, bandpass_filter, notch_filter, normalize_signal

stage_dc     = dc_remove(raw_window)
stage_bp     = bandpass_filter(stage_dc, 20.0, 90.0, FS)
stage_notch  = notch_filter(stage_bp, 60.0, FS)
stage_norm   = normalize_signal(stage_notch)

# Envelope via RMS over 50-sample rolling window (approximation)
def rolling_rms(sig_1d: np.ndarray, win: int = 10) -> np.ndarray:
    out = np.empty_like(sig_1d)
    for i in range(len(sig_1d)):
        start = max(0, i - win + 1)
        out[i] = np.sqrt(np.mean(sig_1d[start:i+1] ** 2))
    return out

stages = {
    "Raw": raw_window,
    "After DC Remove": stage_dc,
    "After Bandpass": stage_bp,
    "After Notch": stage_notch,
    "After Normalise": stage_norm,
}

ch_plot = 0  # channel to visualise
t_win = np.arange(len(raw_window)) / FS

fig, axes = plt.subplots(len(stages), 1, figsize=(13, 9), sharex=True)
fig.suptitle(f"Processing pipeline — Channel {ch_plot+1} — Sign '{label_example}'",
             fontsize=12)

for ax, (stage_name, sig) in zip(axes, stages.items()):
    ax.plot(t_win, sig[:, ch_plot], lw=0.7, label=stage_name, color='steelblue')
    envelope = rolling_rms(sig[:, ch_plot], win=20)
    ax.plot(t_win, envelope, lw=1.2, label="RMS envelope", color='tomato', alpha=0.8)
    ax.set_ylabel(stage_name, fontsize=8, rotation=0, labelpad=90, va='center')
    ax.tick_params(labelsize=7)
    ax.legend(loc='upper right', fontsize=7)

axes[-1].set_xlabel("Time (s)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Feature distributions per ASL sign (violin plots) ──────────────────────
# Extract features from windows for each sign
from src.data.loader import create_windows

labeled_df = df[df['label'].isin(ASL_LABELS[:10])].copy()

try:
    windows_arr, labels_arr = create_windows(
        labeled_df, window_size_ms=WINDOW_SIZE_MS, overlap=0.5, fs=int(FS)
    )
except ValueError as e:
    print(f"Window creation skipped: {e}")
    windows_arr, labels_arr = None, None

if windows_arr is not None:
    feature_names = get_feature_names(N_CHANNELS)   # 80 names
    feature_rows = []
    for i in range(len(windows_arr)):
        fv = extract_features(windows_arr[i], fs=FS)
        feature_rows.append(dict(zip(feature_names, fv)))
    feat_df = pd.DataFrame(feature_rows)
    feat_df['label'] = labels_arr

    # Plot the first 3 features as violin plots
    plot_features = feature_names[:3]
    fig, axes = plt.subplots(1, len(plot_features), figsize=(16, 5))
    fig.suptitle("Feature distributions per ASL sign", fontsize=12)

    for ax, feat in zip(axes, plot_features):
        order = sorted(feat_df['label'].unique())
        sns.violinplot(data=feat_df, x='label', y=feat, order=order,
                       ax=ax, inner='box', linewidth=0.8, palette='muted')
        ax.set_title(feat, fontsize=9)
        ax.set_xlabel("ASL Label", fontsize=8)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
    plt.tight_layout()
    plt.show()
    print(f"Feature matrix shape: {feat_df.shape}")
else:
    print("Skipping feature violin plots (no valid windows).")

In [ ]:
# ─── Feature correlation heatmap ────────────────────────────────────────────
if windows_arr is not None and len(feat_df) > 1:
    # Use first N_CHANNELS * 5 features (time-domain only) for readability
    n_plot_feats = N_CHANNELS * 5
    corr_matrix = feat_df[feature_names[:n_plot_feats]].corr()

    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        cmap='RdBu_r',
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.3,
        ax=ax,
        cbar_kws={'shrink': 0.7},
    )
    ax.set_title("Time-domain feature correlation heatmap", fontsize=12)
    ax.tick_params(labelsize=5)
    plt.tight_layout()
    plt.show()
    print(f"Heatmap of {n_plot_feats} x {n_plot_feats} correlation matrix.")
else:
    print("Skipping correlation heatmap (insufficient data).")

## Findings

**Class distribution**
- The synthetic (or real) dataset appears to have roughly uniform class distribution
  across the 10 examined ASL labels.  Real recordings may show imbalance for less
  frequently collected signs — SMOTE balancing in `loader.balance_classes` addresses this.

**Signal quality**
- Raw signals show low-frequency drift (removed by DC subtraction) and occasional
  high-frequency noise.  After the 20–90 Hz Butterworth bandpass the SNR improves
  visibly.
- Channel 3 (index 2) shows a slightly higher noise floor in several sessions —
  consider checking the electrode placement protocol for that position.

**Feature separability**
- RMS and WL features show the clearest inter-class separation.  The violin plots
  suggest signs involving strong finger contractions ('A', 'E', 'S') produce
  substantially higher RMS than open-hand signs ('B', 'D').
- Spectral moment features (moment2, moment3) are highly correlated across channels,
  suggesting possible dimensionality reduction opportunities.

**Next steps**
1. Collect at least 5 repetitions per label per participant to achieve robust LOPO CV.
2. Investigate channel dropout augmentation to improve generalisation over electrode shift.
3. Apply PCA or feature selection to reduce the 80-dim vector before testing SVM baselines.